# Course 8: Mathematics and AI
## Session 9: Proof assistants: first proofs in Lean 4

**Goal:** state and prove two small theorems about groups and one about categories in Lean 4, and understand what a proof assistant checks that a test cannot.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### Read this first
This notebook's Lean code was **not compiled** when the notebook was made. It follows current Lean 4 and Mathlib conventions, but Mathlib renames lemmas often. Open each snippet in Lean (the web editor at live.lean-lang.org needs no installation), read the error if there is one, and fix it. Doing that is part of the lab.

### 1. Test or proof?
A test checks finitely many cases. Below, Python checks 'the identity is unique' in every group with at most 5 elements, by brute force over multiplication tables. That is strong evidence and it is not a proof: it says nothing about a group with 6 elements, or infinitely many.

In [ ]:
import itertools
def groups_of_order(n):
    """All n x n tables on {0..n-1} that are groups with identity 0 (brute force with pruning by rows)."""
    rows_for = lambda a: [r for r in itertools.permutations(range(n)) if r[0] == a]         # a * 0 = a
    out = []
    def extend(table):
        a = len(table)
        if a == n:
            if all(table[table[x][y]][z] == table[x][table[y][z]] for x in range(n) for y in range(n) for z in range(n)): out.append(table)
            return
        for r in ([tuple(range(n))] if a == 0 else rows_for(a)):
            if all(r[c] != prev[c] for prev in table for c in range(n)): extend(table + [r])
    extend([]); return out

total = 0
for n in range(1, 6):
    G = groups_of_order(n); total += len(G)
    for t in G:
        ids = [e for e in range(n) if all(t[e][a] == a for a in range(n))]
        assert len(ids) == 1                                                             # exactly one left identity
        assert all(sum(t[a][b] == 0 for b in range(n)) == 1 for a in range(n))           # exactly one inverse each
    print(f"order {n}: {len(G)} group tables checked")
print("identity and inverses are unique in all", total, "tables. A proof covers every group at once.")

### 2. The same statements in Lean 4
Create a file `Session9.lean` in a project that depends on Mathlib, or paste into the web editor.

In [ ]:
# NOT RUN and NOT COMPILED here. This is Lean 4 code kept in a Python string so the notebook stays one file.
LEAN_GROUPS = r"""
import Mathlib

-- 1. If e' behaves like an identity on the left, it is the identity.
theorem left_identity_unique {G : Type*} [Group G] (e' : G) (h : ∀ a : G, e' * a = a) : e' = 1 := by
  have h1 : e' * 1 = 1 := h 1
  rw [mul_one] at h1
  exact h1

-- 2. If a * b = 1 then b is the inverse of a. A step-by-step proof with `calc`.
theorem inverse_unique {G : Type*} [Group G] (a b : G) (h : a * b = 1) : b = a⁻¹ := by
  calc b = 1 * b := by rw [one_mul]
    _ = (a⁻¹ * a) * b := by rw [inv_mul_cancel]
    _ = a⁻¹ * (a * b) := by rw [mul_assoc]
    _ = a⁻¹ * 1 := by rw [h]
    _ = a⁻¹ := by rw [mul_one]

-- The same fact is already in Mathlib. `exact?` will find its name for you.
example {G : Type*} [Group G] (a b : G) (h : a * b = 1) : b = a⁻¹ := by
  exact?
"""
print(LEAN_GROUPS)

How to read it:
- `{G : Type*} [Group G]` says: let G be any type with a group structure. The theorem is about all groups.
- After `:=` comes the proof. `by` switches to tactic mode, where each line transforms the goal.
- `rw [mul_one] at h1` rewrites `e' * 1` to `e'` inside the hypothesis `h1`.
- `calc` chains equalities; each step names the fact that justifies it.
- If a lemma name has changed (for example `inv_mul_cancel` was `mul_left_inv` in older Mathlib), Lean says 'unknown identifier'. Type `exact?` or `rw?` at that step and Lean will search for the current name.

### 3. A fact about categories from Mathlib

In [ ]:
# NOT RUN and NOT COMPILED here.
LEAN_CATEGORIES = r"""
import Mathlib
open CategoryTheory

-- Composing with the identity does nothing. In Mathlib, `f ≫ g` means 'f then g'.
example {C : Type*} [Category C] {X Y : C} (f : X ⟶ Y) : 𝟙 X ≫ f = f := by
  simp

-- Inverses of an arrow are unique: if g and g' are both two-sided inverses of f, they are equal.
theorem inv_unique {C : Type*} [Category C] {X Y : C} (f : X ⟶ Y) (g g' : Y ⟶ X)
    (hg : f ≫ g = 𝟙 X) (hg' : g' ≫ f = 𝟙 Y) : g = g' := by
  calc g = 𝟙 Y ≫ g := by rw [Category.id_comp]
    _ = (g' ≫ f) ≫ g := by rw [hg']
    _ = g' ≫ (f ≫ g) := by rw [Category.assoc]
    _ = g' ≫ 𝟙 X := by rw [hg]
    _ = g' := by rw [Category.comp_id]
"""
print(LEAN_CATEGORIES)

Compare `inv_unique` with `inverse_unique` above. It is the same proof. A group is a category with one object in which every arrow has an inverse, so the group theorem is a special case. Seeing that two proofs are one proof is what the language of categories is for.

### 4. What Lean checked, and what it did not
- Checked: every step follows from the axioms and earlier theorems. No case was skipped.
- Not checked: that the statement says what you meant. If you state the wrong theorem, Lean will happily verify the wrong theorem. Session 11 is about that gap.
- `sorry` closes any goal and marks the proof as incomplete. Search for it before you trust a file.

### 5. Why this matters for AI
A language model can write a proof that looks right and is wrong. With a proof assistant in the loop, the model proposes and Lean decides, so a wrong proof is simply rejected. This is the design behind AlphaProof and the open provers used in session 10.

### TODO
1. Compile both snippets. Record every error and how you fixed it.
2. Prove in Lean: in a group, `(a * b)⁻¹ = b⁻¹ * a⁻¹`. First by `simp`, then by hand using `inverse_unique`.
3. Prove: if `a * a = a` in a group, then `a = 1`.
4. State (do not prove) in Lean: 'every group of prime order is cyclic'. Then find it in Mathlib.